In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Importing in annotation file

annot = pd.read_csv('SomalogicAnalyteInfoV1_anonymized.csv')

In [ ]:
# Create a list of proteins to query

mr_proteins = ['DMP1', 'ST13', 'TXNDC15', 'PILRB', 'SERPINF2', 'CTF1',
               'CBLIF', 'PILRA', 'LILRB2', 'APOE', 'ACE', 'DHRS4L2',
               'LRRC37A2', 'SERPINB1', 'ACHE', 'ARHGAP45', 'APOC1',
               'CD2AP', 'BLNK', 'GRN', 'GNAS', 'STX4', 'APP', 'PRSS8']

In [ ]:
# Checking whether MR-identified proteins are present

symbols = annot['entrez_gene_symbol']

mask = symbols.isin(mr_proteins)

In [ ]:
mask.sum()

# 30 matches

In [ ]:
symbols[mask].nunique()

# 21 unique proteins

In [ ]:
# Assessing which MR proteins are unavailable in the dataset

unavailable_proteins = set(mr_proteins) - set(symbols[mask].unique())

In [ ]:
unavailable_proteins

In [ ]:
proteins_of_interest = annot[annot['entrez_gene_symbol'].isin(mr_proteins)]

In [ ]:
proteins_of_interest.sort_values('entrez_gene_symbol')

# Several measurmeents for each protein-
# For now will match with the cohort first


In [ ]:
# Obtaining the ids for proteins of interest

protein_list = list(proteins_of_interest['column_name'])

In [ ]:
protein_list

In [ ]:
# Creating a list of file names to use for dataset access

files = [
    'Somalogic01V1_anonymized.csv',
    'Somalogic02V1_anonymized.csv',
    'Somalogic03V1_anonymized.csv',
    'Somalogic04V1_anonymized.csv',
    'Somalogic05V1_anonymized.csv',
    'Somalogic06V1_anonymized.csv',
    'Somalogic07V1_anonymized.csv',
    'Somalogic08V1_anonymized.csv',
    'Somalogic09V1_anonymized.csv',
    'Somalogic10V1_anonymized.csv',
    'Somalogic11V1_anonymized.csv',
    'Somalogic12V1_anonymized.csv'
]


In [ ]:
# Iterating through files

wanted = ['sample_id'] + protein_list
combined = None
for file in files:
    df = pd.read_csv(file, usecols=lambda c: c in wanted)
    if len(df.columns) <= 1: # if no proteins found skip (aka for first few iterations before they find my proteins)
        continue
    combined = df if combined is None else combined.merge(df, on='sample_id', how='outer')

In [ ]:
# Adding the more informative labels to the columns so I can actually remember what is what

labels = {
    row['column_name']: f"{row['target_full_name']} ({row['entrez_gene_symbol']})"
    for column, row in proteins_of_interest.iterrows()
}

combined = combined.rename(columns=labels)

In [ ]:
combined.to_csv('protein_df.csv', index= False)